In [1]:
import pandas as pd
import numpy as np



tickets = pd.read_csv("../data/raw/tickets.csv")

tickets_clean = tickets.copy()




In [2]:
print("Dataset Shape:", tickets_clean.shape)

print("\nFirst 5 Records:")
display(tickets_clean.head())

print("\nDataset Information:")
tickets_clean.info()

print("\nMissing Values:")
print(tickets_clean.isnull().sum())

print("\nExact Duplicate Records:")
print(tickets_clean.duplicated().sum())

Dataset Shape: (164, 5)

First 5 Records:


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical
3,T0159,C1172,NaN,NaN,none
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ticket_id      164 non-null    str  
 1   customer_id    164 non-null    str  
 2   order_id       156 non-null    str  
 3   text           160 non-null    str  
 4   true_category  164 non-null    str  
dtypes: str(5)
memory usage: 6.5 KB

Missing Values:
ticket_id        0
customer_id      0
order_id         8
text             4
true_category    0
dtype: int64

Exact Duplicate Records:
0


Inspect missing Order IDs

In [3]:
missing_order_id = tickets_clean[
    tickets_clean["order_id"].isna()
]

print("Missing Order ID Records:", len(missing_order_id))

display(missing_order_id)

Missing Order ID Records: 8


,ticket_id,customer_id,order_id,text,true_category
3,T0159,C1172,NaN,NaN,none
10,T0163,C0322,NaN,My delivery was late. The same problem keeps h...,delivery
78,T0162,C0665,NaN,My delivery was late. The same problem keeps h...,delivery
98,T0160,C0670,NaN,NaN,none
128,T0157,C1107,NaN,NaN,none
143,T0164,C0794,NaN,My delivery was late. The same problem keeps h...,delivery
148,T0158,C1106,NaN,NaN,none
158,T0161,C0202,NaN,My delivery was late. The same problem keeps h...,delivery


Inspect missing Ticket Text

In [4]:
missing_ticket_text = tickets_clean[
    tickets_clean["text"].isna()
]

print("Missing Ticket Text Records:", len(missing_ticket_text))

display(missing_ticket_text)

Missing Ticket Text Records: 4


,ticket_id,customer_id,order_id,text,true_category
3,T0159,C1172,NaN,NaN,none
98,T0160,C0670,NaN,NaN,none
128,T0157,C1107,NaN,NaN,none
148,T0158,C1106,NaN,NaN,none


Inspect none Category

In [5]:
none_category = tickets_clean[
    tickets_clean["true_category"].str.lower() == "none"
]

print("Unclassified Category Records:", len(none_category))

display(none_category)

Unclassified Category Records: 4


,ticket_id,customer_id,order_id,text,true_category
3,T0159,C1172,NaN,NaN,none
98,T0160,C0670,NaN,NaN,none
128,T0157,C1107,NaN,NaN,none
148,T0158,C1106,NaN,NaN,none


In [6]:
print("Missing Order IDs:",
      tickets_clean["order_id"].isna().sum())

print("\nMissing Ticket Text:",
      tickets_clean["text"].isna().sum())

print("\nNone Categories:",
      tickets_clean["true_category"].str.lower().eq("none").sum())

Missing Order IDs: 8

Missing Ticket Text: 4

None Categories: 4


In [7]:
display(
    tickets_clean.loc[
        tickets_clean["order_id"].isna()
    ]
)

,ticket_id,customer_id,order_id,text,true_category
3,T0159,C1172,NaN,NaN,none
10,T0163,C0322,NaN,My delivery was late. The same problem keeps h...,delivery
78,T0162,C0665,NaN,My delivery was late. The same problem keeps h...,delivery
98,T0160,C0670,NaN,NaN,none
128,T0157,C1107,NaN,NaN,none
143,T0164,C0794,NaN,My delivery was late. The same problem keeps h...,delivery
148,T0158,C1106,NaN,NaN,none
158,T0161,C0202,NaN,My delivery was late. The same problem keeps h...,delivery


In [8]:
display(
    tickets_clean.loc[
        tickets_clean["true_category"].str.lower().eq("none")
    ]
)

,ticket_id,customer_id,order_id,text,true_category
3,T0159,C1172,NaN,NaN,none
98,T0160,C0670,NaN,NaN,none
128,T0157,C1107,NaN,NaN,none
148,T0158,C1106,NaN,NaN,none


In [9]:
import os

print(os.getcwd())

c:\Users\kamali\OneDrive\Documents\captone\notebook


In [10]:
orders_clean = pd.read_csv("orders_cleaned.csv")
print("Orders Dataset Loaded:", orders_clean.shape)

Orders Dataset Loaded: (12965, 9)


Check customer order history

In [11]:
missing_order_customers = tickets_clean[
    tickets_clean["order_id"].isna()
][["ticket_id", "customer_id", "true_category"]].copy()

customer_order_counts = (
    orders_clean.groupby("customer_id")["order_id"]
    .nunique()
    .reset_index(name="total_orders")
)

customer_order_check = missing_order_customers.merge(
    customer_order_counts,
    on="customer_id",
    how="left"
)

display(customer_order_check)

,ticket_id,customer_id,true_category,total_orders
0,T0159,C1172,none,17
1,T0163,C0322,delivery,19
2,T0162,C0665,delivery,7
3,T0160,C0670,none,8
4,T0157,C1107,none,15
5,T0164,C0794,delivery,2
6,T0158,C1106,none,3
7,T0161,C0202,delivery,29


Handle none Category

In [12]:
tickets_clean["true_category"] = (
    tickets_clean["true_category"].replace("none", pd.NA)
)

In [13]:
print("Missing Categories:",
      tickets_clean["true_category"].isna().sum())

print("\nCategory Distribution:")
print(tickets_clean["true_category"].value_counts(dropna=False))

Missing Categories: 4

Category Distribution:
true_category
delivery     43
fraud        39
billing      39
technical    39
NaN           4
Name: count, dtype: int64


Duplicate Ticket Analysis

In [14]:
print("Exact Duplicate Tickets:",
      tickets_clean.duplicated().sum())

print("\nDuplicate Ticket IDs:",
      tickets_clean["ticket_id"].duplicated().sum())

Exact Duplicate Tickets: 0

Duplicate Ticket IDs: 0


In [15]:
print("Repeated Ticket Messages:",
      tickets_clean["text"].duplicated().sum())

Repeated Ticket Messages: 6


Inspect Repeated Messages

In [16]:
repeated_messages = tickets_clean[
    tickets_clean["text"].notna() &
    tickets_clean["text"].duplicated(keep=False)
].sort_values("text")

display(repeated_messages)

,ticket_id,customer_id,order_id,text,true_category
10,T0163,C0322,NaN,My delivery was late. The same problem keeps h...,delivery
78,T0162,C0665,NaN,My delivery was late. The same problem keeps h...,delivery
143,T0164,C0794,NaN,My delivery was late. The same problem keeps h...,delivery
158,T0161,C0202,NaN,My delivery was late. The same problem keeps h...,delivery


Text Quality Validation

In [17]:
text_whitespace = tickets_clean[
    tickets_clean["text"].notna() &
    (tickets_clean["text"] != tickets_clean["text"].str.strip())
]

print("Messages with Extra Spaces:", len(text_whitespace))

display(text_whitespace)

Messages with Extra Spaces: 0


,ticket_id,customer_id,order_id,text,true_category


Check Empty Messages

In [18]:
empty_messages = tickets_clean[
    tickets_clean["text"].notna() &
    (tickets_clean["text"].str.strip() == "")
]

print("Empty Messages:", len(empty_messages))
display(empty_messages)

Empty Messages: 0


,ticket_id,customer_id,order_id,text,true_category


Validate Category Values

In [19]:
print("Unique Categories:")
print(tickets_clean["true_category"].unique())

print("\nCategory Counts:")
print(tickets_clean["true_category"].value_counts(dropna=False))

Unique Categories:
<StringArray>
['fraud', 'billing', 'technical', nan, 'delivery']
Length: 5, dtype: str

Category Counts:
true_category
delivery     43
fraud        39
billing      39
technical    39
NaN           4
Name: count, dtype: int64


Check Data Types

In [20]:
print("Data Types:")
print(tickets_clean.dtypes)

print("\nDataset Shape:")
print(tickets_clean.shape)

print("\nMissing Values:")
print(tickets_clean.isnull().sum())

Data Types:
ticket_id        str
customer_id      str
order_id         str
text             str
true_category    str
dtype: object

Dataset Shape:
(164, 5)

Missing Values:
ticket_id        0
customer_id      0
order_id         8
text             4
true_category    4
dtype: int64


In [21]:
tickets_clean.to_csv("tickets_cleaned.csv", index=False)

print("Tickets dataset saved successfully!")

Tickets dataset saved successfully!


In [22]:
check_tickets = pd.read_csv("tickets_cleaned.csv")

print("Saved Dataset Shape:", check_tickets.shape)
print("Exact Duplicates:", check_tickets.duplicated().sum())
print("Missing Values:")
print(check_tickets.isnull().sum())

Saved Dataset Shape: (164, 5)
Exact Duplicates: 0
Missing Values:
ticket_id        0
customer_id      0
order_id         8
text             4
true_category    4
dtype: int64


# Tickets Dataset – Data Cleaning Documentation

## 1. Dataset Overview

The Tickets dataset contains customer support ticket information used for customer issue classification and support automation.

- Dataset Name: Tickets
- Original Records: 164
- Original Columns: 5
- Final Records: 164
- Final Columns: 5
- Output File: `tickets_cleaned.csv`

**Column Description:**

- `ticket_id`: Unique identifier for each support ticket.
- `customer_id`: Customer identifier associated with the ticket.
- `order_id`: Related order identifier.
- `text`: Customer support message.
- `true_category`: Actual category of the customer issue.

## 2. Initial Data Quality Audit

The initial data audit identified the following issues:

- Missing `order_id`: 8 records.
- Missing `text`: 4 records.
- Literal `"none"` values in `true_category`: 4 records.
- Exact duplicate rows: 0.
- Duplicate ticket IDs: 0.
- Repeated message values: 6 duplicate flags.

Further investigation showed that four identical non-missing delivery complaints belonged to different customers and tickets. Therefore, these records were retained.

## 3. Data Cleaning Process

### 3.1 Missing Order ID Analysis

- Identified 8 records with missing order IDs.
- Checked the corresponding customer order history using the cleaned Orders dataset.
- Each customer had multiple orders.
- The correct order could not be identified using `customer_id` alone.
- All 8 missing values were retained as NaN to prevent incorrect order mapping.

### 3.2 Missing Ticket Text

- Identified 4 records with missing ticket messages.
- No reliable source was available to recover the original messages.
- Missing values were retained without inserting artificial text.

### 3.3 Category Standardization

- Identified 4 literal `"none"` values in the `true_category` column.
- Converted `"none"` into proper missing values using pandas.
- Verified the remaining category values.

The final valid categories were:
- Delivery: 43 records.
- Fraud: 39 records.
- Billing: 39 records.
- Technical: 39 records.
- Missing categories: 4 records.

No unexpected category values were found.

### 3.4 Duplicate Analysis

- Exact duplicate rows: 0.
- Duplicate ticket IDs: 0.
- Four identical delivery complaint messages were identified.
- These messages belonged to different customers and separate tickets.
- All four records were retained because they were not confirmed duplicate tickets.

### 3.5 Text Quality Validation

- Checked for leading and trailing spaces.
- Messages with extra spaces: 0.
- Checked for empty strings and whitespace-only messages.
- Empty messages: 0.
- No unnecessary modifications were made to the original ticket messages.

### 3.6 Data Type Validation

Verified that all five columns had appropriate string data types:

- `ticket_id` – String.
- `customer_id` – String.
- `order_id` – String.
- `text` – String.
- `true_category` – String.

No data type conversion was required.

## 4. Final Data Quality Validation

The final validation confirmed:

- Total records: 164.
- Total columns: 5.
- Exact duplicate rows: 0.
- Duplicate ticket IDs: 0.
- Missing order IDs: 8.
- Missing ticket messages: 4.
- Missing categories: 4.
- Invalid categories: 0.
- Messages with extra spaces: 0.
- Empty messages: 0.

## 5. Missing Value Treatment

The remaining missing values were intentionally preserved:

- `order_id`: 8 missing values retained because the correct order could not be identified.
- `text`: 4 missing values retained because the original messages were unavailable.
- `true_category`: 4 missing values retained because the actual categories could not be determined.

No arbitrary imputation or unsupported category assignment was performed.

## 6. Output and Verification

The cleaned dataset was saved successfully as `tickets_cleaned.csv`.

The saved file was reloaded and verified.

- Final dataset shape: 164 rows × 5 columns.
- Exact duplicate rows: 0.
- Missing value counts matched the expected cleaning results.
- Data types were appropriate.

## 7. Conclusion

The Tickets dataset underwent data quality auditing, missing value investigation, category standardization, duplicate analysis, text quality validation, and data type verification.

Unrecoverable missing values were retained to maintain data integrity. Similar messages from different customers were also preserved because they represented separate support tickets.

The final dataset, `tickets_cleaned.csv`, is ready for the next stage of the project: **Data Integration and Customer Support Analysis**.

In [23]:
import sqlite3
import pandas as pd
from pathlib import Path

# Find project root
project_root = Path.cwd()

if project_root.name == "notebook":
    project_root = project_root.parent

# Database location
db_folder = project_root / "data"
db_folder.mkdir(exist_ok=True)

db_path = db_folder / "instamart.db"

# Load cleaned CSV files
customers = pd.read_csv(db_folder / "clean" / "customers_cleaned.csv")
products = pd.read_csv(db_folder / "clean" / "products_cleaned.csv")
orders = pd.read_csv(db_folder / "clean" / "orders_cleaned.csv")

# Connect to SQLite database
conn = sqlite3.connect(db_path)

# Create three tables
customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

print("Database created successfully!")
print("Database location:", db_path)

conn.close()

Database created successfully!
Database location: c:\Users\kamali\OneDrive\Documents\captone\data\instamart.db


In [24]:
conn = sqlite3.connect(db_path)

for table in ["customers", "products", "orders"]:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {table}",
        conn
    )
    print(f"{table}: {result.iloc[0, 0]} records")

conn.close()

customers: 1200 records
products: 40 records
orders: 12965 records
